In [ ]:
%%sql -r dataframe_1
USE DATABASE FOOD_DELIVERY_DB;

CREATE SCHEMA IF NOT EXISTS QUARANTINE;

In [ ]:
%%sql -r dataframe_2
use schema quarantine

In [ ]:
%%sql -r dataframe_5
CREATE OR REPLACE TABLE QUARANTINE.BAD_CUSTOMERS_RECORDS
LIKE BRONZE.CUSTOMERS_DETAILS;

In [ ]:
%%sql -r dataframe_3
CREATE OR REPLACE TABLE QUARANTINE.BAD_CUSTOMERS_RECORDS AS
WITH customer_ranked AS (
    SELECT *,
        ROW_NUMBER() OVER (
            PARTITION BY
                customer_id,
                first_name,
                last_name,
                email,
                phone_number,
                date_of_birth,
                gender,
                address_line1,
                city,
                state,
                pincode,
                signup_date,
                customer_segment,
                is_active,
                last_order_date,
                updated_at
            ORDER BY ingestion_ts DESC
        ) AS rn
    FROM BRONZE.CUSTOMERS_DETAILS
)
SELECT *
FROM customer_ranked
WHERE
    customer_id IS NULL
    OR NOT REGEXP_LIKE(
        TRIM(email),
        '^[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}$'
    )
    OR date_of_birth > CURRENT_DATE()
    OR rn > 1;

In [ ]:
%%sql -r dataframe_6
CREATE OR REPLACE TABLE SILVER.CUSTOMER_CLEAN AS
SELECT
    TRIM(customer_id) AS customer_id,
    INITCAP(TRIM(first_name)) AS first_name,
    INITCAP(TRIM(last_name)) AS last_name,
    LOWER(TRIM(email)) AS email,
    TRIM(phone_number) AS phone_number,
    date_of_birth,
    INITCAP(TRIM(gender)) AS gender,
    TRIM(address_line1) AS address_line1,
    INITCAP(TRIM(city)) AS city,
    INITCAP(TRIM(state)) AS state,
    TRIM(pincode) AS pincode,
    signup_date,
    INITCAP(TRIM(customer_segment)) AS customer_segment,
    is_active,
    last_order_date,
    updated_at,
    ingestion_ts,
    source_file_name
FROM BRONZE.CUSTOMERS_DETAILS
WHERE NOT EXISTS (
    SELECT 1
    FROM QUARANTINE.BAD_CUSTOMER_RECORDS q
    WHERE
        COALESCE(BRONZE.CUSTOMERS_DETAILS.customer_id, 'NULL_KEY')
            = COALESCE(q.customer_id, 'NULL_KEY')
        AND COALESCE(BRONZE.CUSTOMERS_DETAILS.email, 'NULL_EMAIL')
            = COALESCE(q.email, 'NULL_EMAIL')
        AND COALESCE(BRONZE.CUSTOMERS_DETAILS.updated_at::STRING, 'NULL_TS')
            = COALESCE(q.updated_at::STRING, 'NULL_TS')
);

In [ ]:
%%sql -r dataframe_7
CREATE OR REPLACE TABLE QUARANTINE.BAD_RESTAURANT_RECORDS AS
SELECT *
FROM BRONZE.RESTAURANT_DETAILS
WHERE
    restaurant_id IS NULL
    OR rating < 0
    OR rating > 5
    OR commission_rate < 0
    OR average_prep_time <= 0
    OR city IS NULL
    OR state IS NULL;

In [ ]:
%%sql -r dataframe_8
CREATE OR REPLACE TABLE SILVER.RESTAURANT_CLEAN AS
SELECT
    TRIM(b.restaurant_id) AS restaurant_id,
    INITCAP(TRIM(b.restaurant_name)) AS restaurant_name,
    INITCAP(TRIM(b.cuisine_type)) AS cuisine_type,
    INITCAP(TRIM(b.city)) AS city,
    INITCAP(TRIM(b.state)) AS state,
    TRIM(b.pincode) AS pincode,
    b.rating,
    b.average_prep_time,
    b.commission_rate,
    b.opening_time,
    b.closing_time,
    b.is_active,
    b.updated_at,
    b.ingestion_ts,
    b.source_file_name
FROM BRONZE.RESTAURANT_DETAILS b
LEFT JOIN QUARANTINE.BAD_RESTAURANT_RECORDS q
    ON COALESCE(b.restaurant_id, 'NULL_ID')
       = COALESCE(q.restaurant_id, 'NULL_ID')
WHERE q.restaurant_id IS NULL;

In [ ]:
%%sql -r dataframe_9
CREATE OR REPLACE TABLE QUARANTINE.BAD_AGENT_RECORDS
LIKE BRONZE.DELIVERY_AGENT_DETAILS;

In [ ]:
%%sql -r dataframe_10
INSERT INTO QUARANTINE.BAD_AGENT_RECORDS
SELECT *
FROM BRONZE.DELIVERY_AGENT_DETAILS
WHERE

    -- mandatory checks
    AGENT_ID IS NULL
    OR AGENT_NAME IS NULL
    OR PHONE_NUMBER IS NULL
    OR CITY IS NULL
    OR VEHICLE_TYPE IS NULL
    OR JOINING_DATE IS NULL

    -- invalid rating
    OR AGENT_RATING NOT BETWEEN 0 AND 5

    -- future joining date
    OR JOINING_DATE > CURRENT_DATE

    -- invalid phone number
    OR NOT REGEXP_LIKE(PHONE_NUMBER, '^[0-9]{10}$')

    -- invalid availability status
    OR UPPER(AVAILABILITY_STATUS)
       NOT IN ('AVAILABLE','BUSY','OFFLINE','ON_DELIVERY')

    -- invalid vehicle type
    OR UPPER(VEHICLE_TYPE)
       NOT IN ('BIKE','SCOOTER','CYCLE','CAR');

In [ ]:
%%sql -r dataframe_11
CREATE OR REPLACE TABLE SILVER.AGENT_CLEAN (

    AGENT_ID VARCHAR,
    AGENT_NAME VARCHAR,
    PHONE_NUMBER VARCHAR,
    CITY VARCHAR,
    VEHICLE_TYPE VARCHAR,
    JOINING_DATE DATE,
    AGENT_RATING FLOAT,
    AVAILABILITY_STATUS VARCHAR,
    UPDATED_AT TIMESTAMP_NTZ,
    INGESTION_TS TIMESTAMP_NTZ,
    SOURCE_FILE_NAME VARCHAR

);

In [ ]:
%%sql -r dataframe_12
INSERT INTO SILVER.AGENT_CLEAN
SELECT DISTINCT

    TRIM(AGENT_ID) AS AGENT_ID,

    INITCAP(TRIM(AGENT_NAME)) AS AGENT_NAME,

    TRIM(PHONE_NUMBER) AS PHONE_NUMBER,

    INITCAP(TRIM(CITY)) AS CITY,

    UPPER(TRIM(VEHICLE_TYPE)) AS VEHICLE_TYPE,

    JOINING_DATE,

    AGENT_RATING,

    UPPER(TRIM(AVAILABILITY_STATUS)) AS AVAILABILITY_STATUS,

    UPDATED_AT,

    INGESTION_TS,

    SOURCE_FILE_NAME

FROM BRONZE.DELIVERY_AGENT_DETAILS

WHERE

    AGENT_ID IS NOT NULL
    AND AGENT_NAME IS NOT NULL
    AND PHONE_NUMBER IS NOT NULL
    AND CITY IS NOT NULL
    AND VEHICLE_TYPE IS NOT NULL
    AND JOINING_DATE IS NOT NULL

    AND AGENT_RATING BETWEEN 0 AND 5

    AND JOINING_DATE <= CURRENT_DATE

    AND REGEXP_LIKE(PHONE_NUMBER, '^[0-9]{10}$')

    AND UPPER(AVAILABILITY_STATUS)
        IN ('AVAILABLE','BUSY','OFFLINE','ON_DELIVERY')

    AND UPPER(VEHICLE_TYPE)
        IN ('BIKE','SCOOTER','CYCLE','CAR');

In [ ]:
%%sql -r dataframe_13
CREATE OR REPLACE TABLE QUARANTINE.BAD_PROMOTION_RECORDS
LIKE BRONZE.PROMOTION_DETAILS;

In [ ]:
%%sql -r dataframe_18
TRUNCATE TABLE QUARANTINE.BAD_PROMOTION_RECORDS;

In [ ]:
%%sql -r dataframe_17
TRUNCATE TABLE SILVER.PROMOTION_CLEAN;

In [ ]:
%%sql -r dataframe_14


In [ ]:
%%sql -r dataframe_42
TRUNCATE TABLE SILVER.PROMOTION_CLEAN;

In [ ]:
%%sql -r dataframe_15
CREATE OR REPLACE TABLE SILVER.PROMOTION_CLEAN (

    PROMO_CODE VARCHAR,
    PROMO_TYPE VARCHAR,
    DISCOUNT_VALUE NUMBER,
    MIN_ORDER_VALUE NUMBER,
    START_DATE DATE,
    END_DATE DATE,
    IS_ACTIVE BOOLEAN,
    UPDATED_AT TIMESTAMP_NTZ,
    INGESTION_TS TIMESTAMP_NTZ,
    SOURCE_FILE_NAME VARCHAR

);

In [ ]:
%%sql -r dataframe_16
INSERT INTO SILVER.PROMOTION_CLEAN
SELECT DISTINCT
    TRIM(PROMO_CODE),
    UPPER(TRIM(PROMO_TYPE)),
    DISCOUNT_VALUE,
    MIN_ORDER_VALUE,
    START_DATE,
    END_DATE,
    IS_ACTIVE,
    UPDATED_AT,
    INGESTION_TS,
    SOURCE_FILE_NAME
FROM BRONZE.PROMOTION_DETAILS
WHERE
    PROMO_CODE IS NOT NULL
    AND PROMO_TYPE IS NOT NULL
    AND DISCOUNT_VALUE > 0
    AND MIN_ORDER_VALUE >= 0
    AND END_DATE >= START_DATE
    AND UPPER(PROMO_TYPE)
        IN ('PERCENTAGE', 'FLAT', 'FREE_DELIVERY');

In [ ]:
%%sql -r dataframe_19
CREATE OR REPLACE TABLE QUARANTINE.BAD_ORDER_RECORDS
LIKE BRONZE.ORDER_DETAILS;

In [ ]:
%%sql -r dataframe_43
DELETE FROM QUARANTINE.BAD_ORDER_RECORDS
WHERE SOURCE_FILE_NAME='Batch_2_Order_Details.csv';

In [ ]:
%%sql -r dataframe_50
TRUNCATE TABLE QUARANTINE.BAD_ORDER_RECORDS;

In [ ]:
%%sql -r dataframe_20
INSERT INTO QUARANTINE.BAD_ORDER_RECORDS
SELECT o.*
FROM BRONZE.ORDER_DETAILS o


LEFT JOIN SILVER.CUSTOMER_CLEAN c
    ON o.CUSTOMER_ID = c.CUSTOMER_ID

LEFT JOIN SILVER.RESTAURANT_CLEAN r
    ON o.RESTAURANT_ID = r.RESTAURANT_ID

LEFT JOIN SILVER.AGENT_CLEAN a
    ON o.AGENT_ID = a.AGENT_ID

LEFT JOIN SILVER.PROMOTION_CLEAN p
    ON o.PROMO_CODE = p.PROMO_CODE

WHERE

    (
    -- mandatory
    o.ORDER_ID IS NULL
    OR o.CUSTOMER_ID IS NULL
    OR o.RESTAURANT_ID IS NULL
    OR o.ORDER_STATUS IS NULL
    OR o.ORDER_PLACED_AT IS NULL

    -- amount checks
    OR o.TOTAL_AMOUNT <= 0
    OR o.DISCOUNT_AMOUNT < 0
    OR o.TAX_AMOUNT < 0
    OR o.FINAL_AMOUNT < 0
    OR o.DELIVERY_FEE < 0
    OR o.DELIVERY_DISTANCE_KM <= 0

    -- timestamp logic
    OR (
        o.ORDER_ACCEPTED_AT IS NOT NULL
        AND o.ORDER_ACCEPTED_AT < o.ORDER_PLACED_AT
    )

    OR (
        o.ORDER_DELIVERED_AT IS NOT NULL
        AND o.ORDER_ACCEPTED_AT IS NOT NULL
        AND o.ORDER_DELIVERED_AT < o.ORDER_ACCEPTED_AT
    )

    -- status validation
    OR UPPER(TRIM(o.ORDER_STATUS))
       NOT IN (
           'PLACED',
           'ACCEPTED',
           'PREPARING',
           'PICKED_UP',
           'DELIVERED',
           'CANCELLED'
       )

    -- source validation
    OR UPPER(TRIM(o.ORDER_SOURCE))
       NOT IN ('ANDROID','IOS','WEB')

    -- business logic
    OR (
        UPPER(TRIM(o.ORDER_STATUS)) = 'DELIVERED'
        AND o.ORDER_DELIVERED_AT IS NULL
    )

    OR (
        UPPER(TRIM(o.ORDER_STATUS)) = 'PLACED'
        AND o.ORDER_ACCEPTED_AT IS NOT NULL
    )

    OR (
        UPPER(TRIM(o.ORDER_STATUS)) = 'PICKED_UP'
        AND o.AGENT_ID IS NULL
    )

    -- referential integrity
    OR c.CUSTOMER_ID IS NULL
    OR r.RESTAURANT_ID IS NULL
    OR (o.AGENT_ID IS NOT NULL AND a.AGENT_ID IS NULL)
    OR (o.PROMO_CODE IS NOT NULL AND p.PROMO_CODE IS NULL)
    );

In [ ]:
%%sql -r dataframe_21


In [ ]:
%%sql -r dataframe_22
INSERT INTO SILVER.ORDER_CLEAN
SELECT DISTINCT

    TRIM(o.ORDER_ID) AS ORDER_ID,
    TRIM(o.CUSTOMER_ID) AS CUSTOMER_ID,
    TRIM(o.RESTAURANT_ID) AS RESTAURANT_ID,
    TRIM(o.AGENT_ID) AS AGENT_ID,

    o.ORDER_PLACED_AT,
    o.ORDER_ACCEPTED_AT,
    o.ORDER_DELIVERED_AT,

    UPPER(TRIM(o.ORDER_STATUS)) AS ORDER_STATUS,

    o.TOTAL_AMOUNT,
    o.DISCOUNT_AMOUNT,
    o.DELIVERY_FEE,
    o.TAX_AMOUNT,
    o.FINAL_AMOUNT,

    o.DELIVERY_DISTANCE_KM,
    o.ESTIMATED_DELIVERY_TIME,
    o.ACTUAL_DELIVERY_TIME,

    INITCAP(TRIM(o.DELIVERY_CITY)) AS DELIVERY_CITY,

    TRIM(o.DELIVERY_PINCODE) AS DELIVERY_PINCODE,

    CASE
        WHEN UPPER(TRIM(o.ORDER_SOURCE)) IN ('ANDROID APP', 'ANDROID')
            THEN 'ANDROID'
        WHEN UPPER(TRIM(o.ORDER_SOURCE)) IN ('IOS')
            THEN 'IOS'
        WHEN UPPER(TRIM(o.ORDER_SOURCE)) = 'WEB'
            THEN 'WEB'
    END AS ORDER_SOURCE,

    TRIM(o.PROMO_CODE) AS PROMO_CODE,

    o.INGESTION_TS,
    o.SOURCE_FILE_NAME

FROM BRONZE.ORDER_DETAILS o

INNER JOIN SILVER.CUSTOMER_CLEAN c
    ON o.CUSTOMER_ID = c.CUSTOMER_ID

INNER JOIN SILVER.RESTAURANT_CLEAN r
    ON o.RESTAURANT_ID = r.RESTAURANT_ID

LEFT JOIN SILVER.AGENT_CLEAN a
    ON o.AGENT_ID = a.AGENT_ID

LEFT JOIN SILVER.PROMOTION_CLEAN p
    ON o.PROMO_CODE = p.PROMO_CODE

WHERE
    o.SOURCE_FILE_NAME = 'Batch_2_Order_Details.csv'

    -- mandatory
    AND o.ORDER_ID IS NOT NULL
    AND o.CUSTOMER_ID IS NOT NULL
    AND o.RESTAURANT_ID IS NOT NULL
    AND o.ORDER_STATUS IS NOT NULL
    AND o.ORDER_PLACED_AT IS NOT NULL

    -- amounts
    AND o.TOTAL_AMOUNT > 0
    AND o.DISCOUNT_AMOUNT >= 0
    AND o.TAX_AMOUNT >= 0
    AND o.FINAL_AMOUNT >= 0
    AND o.DELIVERY_FEE >= 0
    AND o.DELIVERY_DISTANCE_KM > 0

    -- timestamps
    AND (
        o.ORDER_ACCEPTED_AT IS NULL
        OR o.ORDER_ACCEPTED_AT >= o.ORDER_PLACED_AT
    )

    AND (
        o.ORDER_DELIVERED_AT IS NULL
        OR o.ORDER_ACCEPTED_AT IS NULL
        OR o.ORDER_DELIVERED_AT >= o.ORDER_ACCEPTED_AT
    )

    -- status
    AND UPPER(TRIM(o.ORDER_STATUS))
        IN (
            'PLACED',
            'ACCEPTED',
            'PREPARING',
            'PICKED_UP',
            'DELIVERED',
            'CANCELLED'
        )

    -- source
    AND UPPER(TRIM(o.ORDER_SOURCE))
        IN ('ANDROID','ANDROID APP','IOS','WEB')

    -- business logic
    AND NOT (
        UPPER(TRIM(o.ORDER_STATUS)) = 'DELIVERED'
        AND o.ORDER_DELIVERED_AT IS NULL
    )

    AND NOT (
        UPPER(TRIM(o.ORDER_STATUS)) = 'PLACED'
        AND o.ORDER_ACCEPTED_AT IS NOT NULL
    )

    AND NOT (
        UPPER(TRIM(o.ORDER_STATUS)) = 'PICKED_UP'
        AND o.AGENT_ID IS NULL
    )

    -- referential checks
    AND (o.AGENT_ID IS NULL OR a.AGENT_ID IS NOT NULL)
    AND (o.PROMO_CODE IS NULL OR p.PROMO_CODE IS NOT NULL);

In [ ]:
%%sql -r dataframe_33
SELECT SOURCE_FILE_NAME, COUNT(*)
FROM QUARANTINE.BAD_ORDER_RECORDS
GROUP BY SOURCE_FILE_NAME;

In [ ]:
%%sql -r dataframe_23


In [ ]:
%%sql -r dataframe_24
INSERT INTO QUARANTINE.BAD_ORDER_ITEM_RECORDS
SELECT o.*
FROM BRONZE.ORDER_ITEM_DETAILS o

LEFT JOIN SILVER.ORDER_CLEAN ord
    ON o.ORDER_ID = ord.ORDER_ID

WHERE

    o.ORDER_ITEM_ID IS NULL
    OR o.ORDER_ID IS NULL
    OR o.ITEM_NAME IS NULL
    OR o.CATEGORY IS NULL

    OR o.QUANTITY <= 0
    OR o.UNIT_PRICE <= 0
    OR o.TOTAL_PRICE <= 0

    OR o.TOTAL_PRICE <> (o.QUANTITY * o.UNIT_PRICE)

    OR ord.ORDER_ID IS NULL;

In [ ]:
%%sql -r dataframe_25
SELECT *
FROM SILVER.ORDER_CLEAN
WHERE ORDER_ID = 'ORD0000553';

In [ ]:
%%sql -r dataframe_26
CREATE OR REPLACE TABLE SILVER.ORDER_ITEM_CLEAN (

    ORDER_ITEM_ID VARCHAR,
    ORDER_ID VARCHAR,
    ITEM_NAME VARCHAR,
    CATEGORY VARCHAR,
    QUANTITY NUMBER,
    UNIT_PRICE NUMBER,
    TOTAL_PRICE NUMBER,
    IS_VEG BOOLEAN,
    INGESTION_TS TIMESTAMP_NTZ,
    SOURCE_FILE_NAME VARCHAR
);

In [ ]:
%%sql -r dataframe_27
INSERT INTO SILVER.ORDER_ITEM_CLEAN
SELECT DISTINCT

    TRIM(o.ORDER_ITEM_ID) AS ORDER_ITEM_ID,

    TRIM(o.ORDER_ID) AS ORDER_ID,

    INITCAP(TRIM(o.ITEM_NAME)) AS ITEM_NAME,

    INITCAP(TRIM(o.CATEGORY)) AS CATEGORY,

    o.QUANTITY,

    o.UNIT_PRICE,

    o.TOTAL_PRICE,

    o.IS_VEG,

    o.INGESTION_TS,

    o.SOURCE_FILE_NAME

FROM BRONZE.ORDER_ITEM_DETAILS o

INNER JOIN SILVER.ORDER_CLEAN ord
    ON o.ORDER_ID = ord.ORDER_ID

WHERE

    -- mandatory checks
    o.ORDER_ITEM_ID IS NOT NULL
    AND o.ORDER_ID IS NOT NULL
    AND o.ITEM_NAME IS NOT NULL
    AND o.CATEGORY IS NOT NULL

    -- numeric validation
    AND o.QUANTITY > 0
    AND o.UNIT_PRICE > 0
    AND o.TOTAL_PRICE > 0

    -- business validation
    AND o.TOTAL_PRICE = (o.QUANTITY * o.UNIT_PRICE);

In [ ]:
%%sql -r dataframe_28


In [ ]:
%%sql -r dataframe_29


In [ ]:
%%sql -r dataframe_30
INSERT INTO QUARANTINE.BAD_PAYMENT_RECORDS
SELECT p.*
FROM BRONZE.PAYMENT_DETAILS p

LEFT JOIN SILVER.ORDER_CLEAN o
    ON p.ORDER_ID = o.ORDER_ID

WHERE

    p.PAYMENT_ID IS NULL
    OR p.ORDER_ID IS NULL
    OR p.PAYMENT_METHOD IS NULL
    OR p.PAYMENT_STATUS IS NULL
    OR p.PAYMENT_TIMESTAMP IS NULL

    OR p.AMOUNT <= 0
    OR p.REFUND_AMOUNT < 0

    OR UPPER(TRIM(p.PAYMENT_METHOD))
       NOT IN ('UPI','CARD','COD','WALLET')

    OR UPPER(TRIM(p.PAYMENT_STATUS))
       NOT IN ('SUCCESS','FAILED','PENDING','REFUNDED')

    OR (
        p.REFUND_STATUS IS NOT NULL
        AND UPPER(TRIM(p.REFUND_STATUS))
        NOT IN (
            'NONE',
            'INITIATED',
            'FAILED',
            'COMPLETED',
            'REFUNDED'
        )
    )

    OR o.ORDER_ID IS NULL;

In [ ]:
%%sql -r dataframe_31


In [ ]:
%%sql -r dataframe_32
INSERT INTO SILVER.PAYMENT_CLEAN
SELECT DISTINCT

    TRIM(p.PAYMENT_ID),
    TRIM(p.ORDER_ID),
    p.AMOUNT,
    UPPER(TRIM(p.PAYMENT_METHOD)),
    UPPER(TRIM(p.PAYMENT_GATEWAY)),
    UPPER(TRIM(p.PAYMENT_STATUS)),
    p.PAYMENT_TIMESTAMP,
    UPPER(TRIM(p.REFUND_STATUS)),
    p.REFUND_AMOUNT,
    TRIM(p.CARD_LAST4),
    p.INGESTION_TS,
    p.SOURCE_FILE_NAME

FROM BRONZE.PAYMENT_DETAILS p

INNER JOIN SILVER.ORDER_CLEAN o
    ON p.ORDER_ID = o.ORDER_ID

WHERE

    p.PAYMENT_ID IS NOT NULL
    AND p.ORDER_ID IS NOT NULL
    AND p.PAYMENT_METHOD IS NOT NULL
    AND p.PAYMENT_STATUS IS NOT NULL
    AND p.PAYMENT_TIMESTAMP IS NOT NULL

    AND p.AMOUNT > 0
    AND p.REFUND_AMOUNT >= 0

    AND UPPER(TRIM(p.PAYMENT_METHOD))
        IN ('UPI','CARD','NETBANKING','COD','WALLET')

    AND UPPER(TRIM(p.PAYMENT_STATUS))
        IN ('SUCCESS','FAILED','PENDING','REFUNDED')

    AND (
        p.REFUND_STATUS IS NULL
        OR UPPER(TRIM(p.REFUND_STATUS))
        IN ('NONE','INITIATED','COMPLETED','FAILED')
    );

In [ ]:
%%sql -r dataframe_35
SELECT
    CUSTOMER_ID,
    COUNT(*)
FROM SILVER.CUSTOMER_CLEAN
GROUP BY CUSTOMER_ID
HAVING COUNT(*) > 1
LIMIT 20;

In [ ]:
%%sql -r dataframe_46
SELECT

    o.ORDER_ID,

    CASE WHEN c.CUSTOMER_ID IS NULL THEN 'FAIL' ELSE 'PASS' END AS CUSTOMER_CHECK,

    CASE WHEN r.RESTAURANT_ID IS NULL THEN 'FAIL' ELSE 'PASS' END AS RESTAURANT_CHECK,

    CASE
        WHEN o.AGENT_ID IS NOT NULL
         AND a.AGENT_ID IS NULL
        THEN 'FAIL'
        ELSE 'PASS'
    END AS AGENT_CHECK,

    CASE
        WHEN o.PROMO_CODE IS NOT NULL
         AND p.PROMO_CODE IS NULL
        THEN 'FAIL'
        ELSE 'PASS'
    END AS PROMO_CHECK

FROM BRONZE.ORDER_DETAILS o

LEFT JOIN SILVER.CUSTOMER_CLEAN c
    ON o.CUSTOMER_ID = c.CUSTOMER_ID

LEFT JOIN SILVER.RESTAURANT_CLEAN r
    ON o.RESTAURANT_ID = r.RESTAURANT_ID

LEFT JOIN SILVER.AGENT_CLEAN a
    ON o.AGENT_ID = a.AGENT_ID

LEFT JOIN SILVER.PROMOTION_CLEAN p
    ON o.PROMO_CODE = p.PROMO_CODE

WHERE o.ORDER_ID='ORD_BATCH2_0034';

In [ ]:
%%sql -r dataframe_47
SELECT

COUNT(CASE WHEN c.CUSTOMER_ID IS NULL THEN 1 END) AS BAD_CUSTOMER,

COUNT(CASE WHEN r.RESTAURANT_ID IS NULL THEN 1 END) AS BAD_RESTAURANT,

COUNT(
    CASE
        WHEN o.AGENT_ID IS NOT NULL
         AND a.AGENT_ID IS NULL
        THEN 1
    END
) AS BAD_AGENT,

COUNT(
    CASE
        WHEN o.PROMO_CODE IS NOT NULL
         AND p.PROMO_CODE IS NULL
        THEN 1
    END
) AS BAD_PROMO

FROM BRONZE.ORDER_DETAILS o

LEFT JOIN SILVER.CUSTOMER_CLEAN c
    ON o.CUSTOMER_ID = c.CUSTOMER_ID

LEFT JOIN SILVER.RESTAURANT_CLEAN r
    ON o.RESTAURANT_ID = r.RESTAURANT_ID

LEFT JOIN SILVER.AGENT_CLEAN a
    ON o.AGENT_ID = a.AGENT_ID

LEFT JOIN SILVER.PROMOTION_CLEAN p
    ON o.PROMO_CODE = p.PROMO_CODE

WHERE o.SOURCE_FILE_NAME='Batch_2_Order_Details.csv';

In [ ]:
%%sql -r dataframe_48
SELECT RESTAURANT_ID FROM SILVER.RESTAURANT_CLEAN LIMIT 20;


In [ ]:
%%sql -r dataframe_49
SELECT AGENT_ID FROM SILVER.AGENT_CLEAN ;

In [ ]:
%%sql -r dataframe_51
SELECT COUNT(*) FROM SILVER.ORDER_CLEAN;

